# Docker 第2周：Dockerfile — 构建自己的镜像

> **学习目标**：能将任意 Python 应用打包成 Docker 镜像，掌握构建优化技巧

---

## 为什么需要 Dockerfile？

上一周你用的是别人做好的镜像（nginx、redis、python）。这周你要学会**自己造镜像**。

Dockerfile 就是一个**构建脚本**——里面写满了"怎么组装这个镜像"的指令。

类比：
- `docker pull nginx` = 去超市买现成的蛋糕
- `docker build -t my-app .` = 按照食谱（Dockerfile）自己烤一个蛋糕

### Dockerfile 的本质

Dockerfile 中的每条指令都会生成一个新的**镜像层（layer）**。层是只读的，叠在一起构成最终镜像。

```
FROM python:3.12-slim     → 层 0：拿基础镜像
WORKDIR /app              → 层 1：设置工作目录（元数据，几乎无体积）
COPY requirements.txt .   → 层 2：复制依赖文件
RUN pip install -r ...    → 层 3：安装依赖（这一层通常最大）
COPY . .                  → 层 4：复制应用代码
CMD ["python", "app.py"]  → 层 5：设置启动命令（元数据）
```

**层数越多镜像越大吗？** 不一定。层存储的是差异（diff）。如果第 3 层安装了 200MB 的包，第 4 层只加了 2KB 的代码，总镜像 ≈ 200MB + 2KB。关键是**每一层只增加相对于上一层的差异**。

---

## 核心指令详解

### `FROM` — 指定基础镜像

每个 Dockerfile 的第一条指令（除了 `ARG`）。

```dockerfile
FROM python:3.12-slim
```

**常见 Python 基础镜像对比**：

| 镜像 tag | 大小 | 适用场景 |
|----------|------|----------|
| `python:3.12` | ~1GB | 完整 Debian，啥都有，但很大 |
| `python:3.12-slim` | ~150MB | 精简版 Debian，**开发/生产首选** |
| `python:3.12-alpine` | ~50MB | 基于 Alpine Linux（musl libc），**最小但可能有兼容问题** |

> **建议**：先用 `slim`，如果需要极致压缩体积再考虑 `alpine`。Alpine 用 musl libc 而不是 glibc，有些 Python 包（尤其是 C 扩展）在上面可能编不过。

### `RUN` — 构建时执行命令

```dockerfile
RUN pip install --no-cache-dir -r requirements.txt
```

`RUN` 在**构建时**执行，结果被提交为一个新的镜像层。

**注意**：每条 `RUN` 会生成一个新层。所以要把相关命令合并在一个 `RUN` 里：

```dockerfile
# 不好：生成 3 个层
RUN apt-get update
RUN apt-get install -y gcc
RUN rm -rf /var/lib/apt/lists/*

# 好：合并在一个层，最后清理减小体积
RUN apt-get update && \
    apt-get install -y gcc && \
    rm -rf /var/lib/apt/lists/*
```

### `COPY` — 从宿主机复制文件

```dockerfile
COPY requirements.txt /app/
COPY . /app/
```

`COPY` 只能复制**构建上下文**（`docker build` 时指定的目录）中的文件。

### `WORKDIR` — 设置工作目录

```dockerfile
WORKDIR /app
```

如果目录不存在会自动创建。之后的 `RUN`、`CMD`、`COPY . .` 都以此为当前目录。

### `CMD` vs `ENTRYPOINT`

| 指令 | 作用 | 可被覆盖？ |
|------|------|-----------|
| `CMD` | 容器启动时的**默认**命令 | 可以被 `docker run ... <command>` 覆盖 |
| `ENTRYPOINT` | 容器的**入口点** | 不被覆盖，`docker run` 的参数会追加到后面 |

实践建议：用 `ENTRYPOINT` 指定主程序，用 `CMD` 指定默认参数。

In [ ]:
# 对比 CMD 和 ENTRYPOINT 的行为差异
# 做一个简单的实验

# 镜像 A：只有 CMD
! echo 'FROM alpine
CMD ["echo", "我是默认消息"]' | docker build -t cmd-test -

print("=== 镜像 A: CMD ===")
! docker run --rm cmd-test
! docker run --rm cmd-test echo "覆盖了"

In [ ]:
# 镜像 B：ENTRYPOINT + CMD
! echo 'FROM alpine
ENTRYPOINT ["echo"]
CMD ["我是默认消息"]' | docker build -t entrypoint-test -

print("=== 镜像 B: ENTRYPOINT + CMD ===")
! docker run --rm entrypoint-test
# docker run 后面的参数会被追加到 ENTRYPOINT 后面，而不是替换
! docker run --rm entrypoint-test "我覆盖了默认消息"

### 其他常用指令

| 指令 | 作用 |
|------|------|
| `EXPOSE 8000` | 声明容器监听的端口（文档作用，不实际映射） |
| `ENV NAME=value` | 设置环境变量，运行时也可改 |
| `ARG NAME=value` | 构建参数，只在构建时存在 |
| `USER nobody` | 以哪个用户运行（安全：别用 root） |
| `VOLUME /data` | 声明匿名卷（建议用 compose 管理卷，更显式） |
| `HEALTHCHECK` | 定义健康检查命令 |

---

## 实战：将 Python 脚本打包成镜像

我们从最简单的开始——打包一个单文件 Python 脚本。

In [ ]:
# 1. 准备一个简单的 Python 应用
! mkdir -p /tmp/docker-demo

%%writefile /tmp/docker-demo/hello.py
import sys
import datetime

def main():
    name = sys.argv[1] if len(sys.argv) > 1 else "World"
    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{now}] Hello, {name}!")

if __name__ == "__main__":
    main()

In [ ]:
# 2. 写 Dockerfile
%%writefile /tmp/docker-demo/Dockerfile
# 基础镜像
FROM python:3.12-slim

# 设置工作目录（不存在会自动创建）
WORKDIR /app

# 复制应用代码
COPY hello.py .

# 容器启动时执行的命令
# ENTRYPOINT 用列表形式（exec form），避免 shell 解析问题
ENTRYPOINT ["python", "hello.py"]

# 默认参数（可被覆盖）
CMD ["Docker"]

In [ ]:
# 3. 构建镜像
! docker build -t hello-app:1.0 /tmp/docker-demo

# 4. 查看构建好的镜像
! docker images | grep hello-app

In [ ]:
# 5. 运行容器
# 不传参数，使用默认值
! docker run --rm hello-app:1.0

# 传入自定义参数（覆盖 CMD）
! docker run --rm hello-app:1.0 Pythonista

# 完全覆盖启动命令
! docker run --rm --entrypoint python hello-app:1.0 -c "print('完全覆盖')

---

## 构建缓存：加速迭代

Docker 构建时，会按顺序检查每条指令。如果某条指令及其上下文没变，就直接用缓存。

**关键原则：把不常变的放在前面，常变的放在后面。**

```dockerfile
# 好的顺序
FROM python:3.12-slim          # 几乎不变
WORKDIR /app                    # 几乎不变
COPY requirements.txt .        # 依赖文件很少变
RUN pip install -r requirements.txt  # 依赖变了才重装
COPY . .                       # 代码经常变 ← 放最后！
```

这样改了代码只重建最后两层（COPY 代码 + CMD），依赖安装层用缓存，秒级构建。

In [ ]:
# 演示：构建一个带依赖的 Python 应用
%%writefile /tmp/docker-demo/requirements.txt
flask==3.0.0

%%writefile /tmp/docker-demo/app.py
from flask import Flask
app = Flask(__name__)

@app.route("/")
def home():
    return "<h1>Hello from Docker!</h1>"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
%%writefile /tmp/docker-demo/Dockerfile.flask
FROM python:3.12-slim

WORKDIR /app

# 先复制依赖文件（不常变）
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 再复制代码（常变）
COPY app.py .

EXPOSE 5000
CMD ["python", "app.py"]

In [ ]:
# 第一次构建（全量，慢）
! docker build -t flask-app:1.0 -f /tmp/docker-demo/Dockerfile.flask /tmp/docker-demo

In [ ]:
# 修改 app.py（模拟改代码），再构建一次
%%writefile /tmp/docker-demo/app.py
from flask import Flask
app = Flask(__name__)

@app.route("/")
def home():
    return "<h1>Hello from Docker! (v2)</h1>"

@app.route("/health")
def health():
    return {"status": "ok"}

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

# 第二次构建——注意看日志，前几步显示 Using cache
! docker build -t flask-app:2.0 -f /tmp/docker-demo/Dockerfile.flask /tmp/docker-demo
print("\n注意上面的输出：pip install 那步显示 'Using cache'，秒过！")

---

## 多阶段构建：编译和运行分离

有些场景需要编译工具（gcc、Rust、Go），但运行时不需要。如果全放一个镜像里，会很臃肿。

**多阶段构建**：在一个 Dockerfile 里定义多个 `FROM`，最终镜像只保留最后一阶段的内容。

```dockerfile
# 第1阶段：编译
FROM python:3.12 AS builder
COPY requirements.txt .
RUN pip install --user -r requirements.txt

# 第2阶段：运行（最终镜像）
FROM python:3.12-slim
COPY --from=builder /root/.local /root/.local
COPY . .
CMD ["python", "app.py"]
```

最终镜像不包含 builder 阶段的 gcc 等编译工具，体积大幅缩减。

In [ ]:
# 对比：单阶段 vs 多阶段构建的体积

# 单阶段
%%writefile /tmp/docker-demo/Dockerfile.single
FROM python:3.12
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY app.py .
CMD ["python", "app.py"]

! docker build -t flask-single -f /tmp/docker-demo/Dockerfile.single /tmp/docker-demo -q

# 多阶段
%%writefile /tmp/docker-demo/Dockerfile.multi
# Stage 1: 安装依赖
FROM python:3.12-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# Stage 2: 运行
FROM python:3.12-slim
WORKDIR /app
# 从 builder 阶段复制已安装的包
COPY --from=builder /root/.local /usr/local
COPY app.py .
CMD ["python", "app.py"]

! docker build -t flask-multi -f /tmp/docker-demo/Dockerfile.multi /tmp/docker-demo -q

# 对比大小
! docker images flask-single flask-multi --format "table {{.Repository}}\t{{.Tag}}\t{{.Size}}"

---

## .dockerignore：排除不需要的文件

类似 `.gitignore`，告诉 Docker 构建时忽略某些文件。

没有 `.dockerignore` 的话，`COPY . .` 会把 `__pycache__`、`.git`、`.venv` 全复制进去——又慢又大还可能出错。

```
# .dockerignore 示例
__pycache__/
*.pyc
.git/
.venv/
venv/
*.md
.env
*.log
.DS_Store
```

---

## 安全最佳实践

### 1. 非 root 运行

Docker 默认以 root 运行容器。如果应用有漏洞被攻破，攻击者就拿到了 root 权限。

```dockerfile
FROM python:3.12-slim

# 创建非 root 用户
RUN useradd --create-home --shell /bin/bash appuser

WORKDIR /app
COPY --chown=appuser:appuser . .

# 切换到非 root 用户
USER appuser

CMD ["python", "app.py"]
```

### 2. 镜像瘦身清单

- 用 `slim` 或 `alpine` 基础镜像
- `pip install --no-cache-dir` 不缓存下载的包
- 在同一个 `RUN` 中清理 apt 缓存：`rm -rf /var/lib/apt/lists/*`
- 用多阶段构建
- 写 `.dockerignore`

### 3. HEALTHCHECK

让 Docker 知道容器是否真正"健康"（不只是进程在跑）：

```dockerfile
HEALTHCHECK --interval=30s --timeout=3s --retries=3 \
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:5000/health')"
```

---

## 镜像仓库：发布你的镜像

构建好的镜像需要发布到 Registry（镜像仓库）才能被其他人或服务器拉取。

### Docker Hub 发布流程

```bash
# 1. 登录
docker login

# 2. 打标签（必须包含你的用户名）
docker tag flask-app:1.0 your-username/flask-app:1.0

# 3. 推送
docker push your-username/flask-app:1.0

# 4. 其他机器拉取
docker pull your-username/flask-app:1.0
```

In [ ]:
# 打 tag 演示（不实际推送）
! docker tag flask-app:1.0 flask-app:latest
! docker tag flask-app:1.0 flask-app:2024-01-01
! docker images | grep flask-app

# 注意：多个 tag 指向同一个 IMAGE ID，它们本质是同一个镜像

---

## 🎯 第2周总结

| 指令 | 作用 | 关键点 |
|------|------|--------|
| `FROM` | 基础镜像 | 选 slim，不要用 latest |
| `RUN` | 构建时执行 | 合并命令，清理缓存 |
| `COPY` | 复制文件 | 先复制依赖再复制代码（缓存） |
| `WORKDIR` | 工作目录 | 比 `RUN cd` 好 |
| `CMD` | 默认启动命令 | 可被覆盖 |
| `ENTRYPOINT` | 入口点 | 不被覆盖 |
| `ENV` | 环境变量 | 运行时配置 |
| `USER` | 运行用户 | 别用 root |

### 写 Dockerfile 的思维模型

把 Dockerfile 想象成**给一台空白电脑装系统**：
1. 先装操作系统（`FROM`）
2. 装运行时环境（`RUN apt-get` / `pip install`）
3. 把你的代码放进去（`COPY`）
4. 告诉它开机后执行什么（`CMD`）

---

## 🧪 综合练习：容器化一个 CLI Todo 应用

为下面的 Todo 应用编写 Dockerfile，要求：

1. 基础镜像用 `python:3.12-slim`
2. 非 root 用户运行
3. 不硬编码数据路径——通过卷挂载
4. 多阶段构建（如果有编译需求）或优化层缓存
5. 写 `.dockerignore` 排除不必要的文件
6. 构建后测试：增删改查操作都能正常工作

In [ ]:
# 练习：Todo CLI 应用源码
%%writefile /tmp/docker-demo/todo.py
import sys
import json
import os
from datetime import datetime

DATA_FILE = os.environ.get("TODO_FILE", "/data/todos.json")

def load():
    if not os.path.exists(DATA_FILE):
        return []
    with open(DATA_FILE) as f:
        return json.load(f)

def save(todos):
    os.makedirs(os.path.dirname(DATA_FILE), exist_ok=True)
    with open(DATA_FILE, 'w') as f:
        json.dump(todos, f, ensure_ascii=False, indent=2)

def cmd_list():
    todos = load()
    if not todos:
        print("暂无待办事项")
        return
    for i, t in enumerate(todos, 1):
        status = "✓" if t["done"] else "□"
        print(f"{i}. [{status}] {t['title']} ({t['created_at']})")

def cmd_add(title):
    todos = load()
    todos.append({
        "title": title,
        "done": False,
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M")
    })
    save(todos)
    print(f"已添加: {title}")

def cmd_done(index):
    todos = load()
    if 0 < index <= len(todos):
        todos[index - 1]["done"] = True
        save(todos)
        print(f"已完成: {todos[index - 1]['title']}")
    else:
        print(f"无效序号: {index}")

def cmd_remove(index):
    todos = load()
    if 0 < index <= len(todos):
        removed = todos.pop(index - 1)
        save(todos)
        print(f"已删除: {removed['title']}")
    else:
        print(f"无效序号: {index}")

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("用法: python todo.py [list|add|done|remove] [args]")
        sys.exit(1)
    
    command = sys.argv[1]
    if command == "list":
        cmd_list()
    elif command == "add" and len(sys.argv) > 2:
        cmd_add(" ".join(sys.argv[2:]))
    elif command == "done" and len(sys.argv) > 2:
        cmd_done(int(sys.argv[2]))
    elif command == "remove" and len(sys.argv) > 2:
        cmd_remove(int(sys.argv[2]))
    else:
        print(f"未知命令: {command}")

# 你的 Dockerfile 写在这里
# %%writefile /tmp/docker-demo/Dockerfile.todo
# ...

pass